# Exercise B
## Turning your time-dependent model into a Gammapy model

This notebook adapts [this gammapy tutorial](https://docs.gammapy.org/2.1/tutorials/analysis-3d/event_sampling_nrg_depend_models.html#create-the-energy-dependent-temporal-model) but instead of different PowerLaw models, we will use a GAMERA model

## A note on units
Units in GAMERA are mostly in cgs, with several exceptions that are meaningful in the astrophysical context.

    times are in years, except for particle escape time scales
    lengths are in parsecs
    when retrieving SEDs, for convenience the units are
        erg / s / cm2 vs TeV for radiation flux SEDs
        erg / s vs. TeV for luminosity SEDs (i.e. when the distance to the source is not set)
        erg vs. TeV for particle SEDs

    (differential spectra are again in cgs units)
    gas densities are in number densities.

Also, the units of the spectrum will impact the unit of the calculated radiation spectra. For instance, if the unit of the particle spectrum is only differential in energy, i.e. 1/erg, the output radiation spectra will have the unit of a flux if a source distance is specified (see below) or differential photon count per energy and time if not.

On the other hand, if the input unit is differential also in volume, i.e. 1/erg/cm3, then also the output radiation spectrum will be. Therefore, if no distance is specified in this case, a volume photon emissivity will be calculated, i.e. d3N/dEdVdt.

In [ ]:
import gappa as gp
import numpy as np
import astropy.units as u
from gammapy.maps import MapAxis
import matplotlib.pyplot as plt


## We will use the temporal model which we constructed in exercise A and turn it into a Gammapy time-dependent model

Using the same B field definition

In [ ]:
def B_field(t,...):
    return (...).to('G') # replace by yours  from exercise A

# Define array of times
times = n  # replace by yours from exercise A

# Calculate B
B = B_field(times,...) # replace by yours from exercise A

# Construct lookup
B_field_lookup = np.array(list(zip(times.to_value('yr'), B.to_value('G'))))


and the same particle spectrum

In [ ]:
def define_injected_spectrum(...)
# replace by yours from exercise A

we now put it all together into one function

In [ ]:
def photon_spectrum_at_t(t=10*u.day, photon_energies=np.logspace(-16,3,200)*u.TeV):
    # Set up the Particles class
    fp = gp.Particles()
    fp.ToggleQuietMode()

    # Provide B field lookup
    fp.XXX(B_field_lookup) # Insert correct GAMERA function

    # Set photon target
    fp.XX(T,edens) # Insert correct GAMERA function and define T and edens

    # Change minimum time
    tmin = times[0].to_value('yr')/2
    fp.SetTmin(tmin)

    # Set injected spectrum
    power_law_spectrum = define_injected_spectrum()
    fp.XXX(power_law_spectrum)  # Insert correct GAMERA function

    # change t to years
    t = t.to_value('yr')
    print("Time", t)

    # Set Age to t
    fp.XXX(t)  # Insert correct GAMERA function (easy ;-) )

    # CalculateElectronSpectrum
    fp.XXX()  # Insert correct GAMERA function (easy ;-) )

    # Set up radiation class 
    fr = gp.Radiation()
    fr.ToggleQuietMode()

    # Set photon target
    fr.XXX(T, edens) # similar to above

    # Set distance
    distance = 1*u.kpc # customize!
    fr.SetDistance(distance.to_value('pc'))


    # Get B field at this time and set it
    B = fp.GetBField(t)
    fr.SetBField(B)

    # SetElectrons
    fr.SetElectrons(fp.GetParticleSpectrum())

    # compute the spectrum on the points
    fr.XXXX(photon_energies.to_value('erg')) # Insert correct GAMERA function

    total_sed = np.asarray(fr.GetTotalSED())
    
    # if the energy requested is outside of the range of the SEDs we just return 0
    if len(total_sed)==0:
        return np.zeros(len(photon_energies))*u.cm**-2*u.eV**-1*u.s**-1

    sed = total_sed[:,1]

    # Get back to the world of units
    sed *= u.erg * u.cm**-2 *u.s **-1

    result = (sed / ((total_sed[:,0]*u.TeV) ** 2)).to("1 / (cm2 eV s)")
    result = np.reshape(result, photon_energies.shape)
    return result

In [ ]:
# Test it
photon_spectrum_at_t()

### From now the steps are fully following [this gammapy tutorial](https://docs.gammapy.org/2.1/tutorials/analysis-3d/event_sampling_nrg_depend_models.html#create-the-energy-dependent-temporal-model) but adapted to our case

Define a RegionNDMap

In [ ]:
from gammapy.maps import RegionNDMap
from regions import PointSkyRegion
from astropy.coordinates import SkyCoord
from astropy.time import Time
from gammapy.modeling.models import LightCurveTemplateTemporalModel

In [ ]:
# Source position
pointing_position = SkyCoord(....) # Customize

# Time axis 
time_axis = MapAxis.from_bounds(...) # use similar range as the model, not too many bins if you want it to be fast

# Energy axis
energy_axis = MapAxis.from_energy_bounds(
    energy_min=, energy_max=, nbin=
) # customize


# RegionNDMap
m = RegionNDMap.create(
    region=PointSkyRegion(center=pointing_position),
    axes=[energy_axis, time_axis],
    unit="cm-2s-1TeV-1",
)

to compute the spectra as a function of time we extract the coordinates of the geometry

In [ ]:
coords = # extract coordinates

evaluate the spectra and fill the RegionNDMap

In [ ]:
energies = coords["energy"].flatten()
for idx, time in enumerate(coords["time"].flatten()):
    model = photon_spectrum_at_t(time, energies)
    m.data[idx, :,0,0] = model

## Turn RegionNDMap into TemplateTemporalModel

In [ ]:
t_ref = Time(51544.00074287037, format="mjd", scale="tt") # random time here but in reality it would be your observation
filename = "./temporal_model_map.fits"
temporal_model  = LightCurveTemplateTemporalModel(m, t_ref=t_ref, filename=filename) # create temportal model
#temporal_model.write(filename, format="map", overwrite=True)

And plot at different energies

In [ ]:
time_range = temporal_model.reference_time + [0, 100] * u.day # adapt to your timescales

temporal_model.plot(time_range=time_range, energy=[0.1, 0.5, 1, 5] * u.TeV) # adapt to your energy range
plt.legend()
plt.semilogy()
plt.show()